In [ ]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

In [ ]:
class State(TypedDict):
    dish: str
    ingredients: list[dict]
    recipe_steps: str
    plating_instructions: str

class Ingredient(BaseModel):
    name: str
    quantity: str
    unit: str

class IngredientsOutput(BaseModel):
    ingredients: List[Ingredient]

In [ ]:
def list_ingredients(state: State):
    structured_llm = llm.with_structured_output(IngredientsOutput)
    response = structured_llm.invoke(f"{state['dish']}을 만드는 데 필요한 재료 5~8가지만 알려줘.")  
    return {"ingredients": response.ingredients}

def create_recipe(state: State):
    response = llm.invoke(f"{state['ingredients']}을(를) 사용하여 {state['dish']}을(를) 만드는 단계별 요리법을 작성해줘")
    return {
        "recipe_steps": response.content
    }

def describe_plating(state: State):
    response = llm.invoke(f"이 레시피 {state["recipe_steps"]}를 바탕으로 {state["dish"]}를 예쁘게 플레이팅하는 방법을 설명해")
    return {
        "plating_instructions": response.content
    }

def gate(state: State): 
    ingredients = state["ingredients"]

    if len(ingredients) > 8 or len(ingredients) <5:
        return False
    
    return True

In [ ]:
graph_builder = StateGraph(State)

graph_builder.add_node("list_ingredients", list_ingredients)
graph_builder.add_node("create_recipe", create_recipe)
graph_builder.add_node("describe_plating", describe_plating)

graph_builder.add_edge(START, "list_ingredients")
graph_builder.add_conditional_edges("list_ingredients", gate, { 
    True : "create_recipe",
    False: "list_ingredients"
})
graph_builder.add_edge("create_recipe", "describe_plating")
graph_builder.add_edge("describe_plating", END)

graph = graph_builder.compile()

In [ ]:
graph

In [ ]:
#graph.invoke({"dish": "김치찌개"})